# Can we predict hospital readmission?
Dataset: diabetic-patient hospital encounters (~100k rows), target `readmitted` in {`NO`, `>30`, `<30`}.

**Approach**
- Primary question: predict readmission **within 30 days** (`<30` vs everything else). A secondary run predicts **any** readmission.
- Split **by patient** (`patient_nbr`) so the same person never appears in both train and test (otherwise scores are inflated).
- Drop encounters ending in death/hospice (those patients cannot be readmitted).
- Compare a trivial baseline, logistic regression and gradient boosting using ROC-AUC and PR-AUC (classes are imbalanced).

In [ ]:
import os, warnings
os.environ.setdefault('MPLCONFIGDIR', os.environ.get('TMPDIR', '/tmp'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
SEED = 0
df = pd.read_csv('data.csv', na_values=['?'], low_memory=False)
df.shape

## 1. Look at the data

In [ ]:
print(df.readmitted.value_counts(), '\n')
print(df.readmitted.value_counts(normalize=True).round(3), '\n')
print('encounters:', len(df), ' unique patients:', df.patient_nbr.nunique())
miss = df.isna().mean(); miss[miss > 0].sort_values(ascending=False).to_frame('missing share')

`weight` (97% missing) is dropped. `max_glu_serum` (95%) and `A1Cresult` (83%) are mostly missing, but *missing* means "not measured", which may be informative, so it is kept as its own level. `payer_code` and `medical_specialty` get an explicit `Missing` level too.

## 2. Cleaning and feature engineering

In [ ]:
d = df.copy()
# Discharged to death / hospice -> cannot be readmitted (IDs 11,13,14,19,20,21)
dead = d.discharge_disposition_id.isin([11, 13, 14, 19, 20, 21])
print('dropping', dead.sum(), 'expired/hospice encounters')
d = d[~dead & (d.gender != 'Unknown/Invalid')].copy()

d['y30'] = (d.readmitted == '<30').astype(int)
d['yany'] = (d.readmitted != 'NO').astype(int)

def icd_group(x):
    if pd.isna(x): return 'Missing'
    if str(x)[0] in 'VE': return 'Other'
    v = float(x)
    if 390 <= v <= 459 or int(v) == 785: return 'Circulatory'
    if 460 <= v <= 519 or int(v) == 786: return 'Respiratory'
    if 520 <= v <= 579 or int(v) == 787: return 'Digestive'
    if int(v) == 250: return 'Diabetes'
    if 800 <= v <= 999: return 'Injury'
    if 710 <= v <= 739: return 'Musculoskeletal'
    if 580 <= v <= 629 or int(v) == 788: return 'Genitourinary'
    if 140 <= v <= 239: return 'Neoplasms'
    return 'Other'
for k in ['diag_1', 'diag_2', 'diag_3']:
    d[k + '_grp'] = d[k].map(icd_group)
    d = d.drop(columns=k)

d['age_mid'] = d.age.str.extract(r'\[(\d+)-')[0].astype(int) + 5
d['n_prior_visits'] = d.number_outpatient + d.number_emergency + d.number_inpatient
meds = ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride','acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone','rosiglitazone','acarbose','miglitol','troglitazone','tolazamide','examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin','glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone']
d['n_meds_used'] = (d[meds] != 'No').sum(axis=1)
d['n_meds_changed'] = d[meds].isin(['Up', 'Down']).sum(axis=1)

# ID columns that are really categories
for k in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
    d[k] = d[k].astype(str)
for k in ['max_glu_serum', 'A1Cresult', 'race', 'payer_code', 'medical_specialty']:
    d[k] = d[k].fillna('Missing').astype(str)
# rare levels -> 'Other'
for k in ['medical_specialty', 'payer_code', 'discharge_disposition_id', 'admission_source_id']:
    vc = d[k].value_counts(); d[k] = d[k].where(d[k].isin(vc[vc >= 200].index), 'Other')

const = [m for m in meds if d[m].nunique() <= 1]
drop = ['encounter_id', 'weight', 'readmitted', 'y30', 'yany', 'age', 'patient_nbr'] + const
groups = d.patient_nbr
X = d.drop(columns=drop)
cat = X.select_dtypes('object').columns.tolist()
num = [x for x in X.columns if x not in cat]
print(len(cat), 'categorical,', len(num), 'numeric features;', len(X), 'rows; constant meds dropped:', const)

## 3. Quick exploration

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
d.groupby('number_inpatient').y30.agg(['mean', 'size']).query('size>=100')['mean'].plot.bar(ax=ax[0], color='#3b6fb6'); ax[0].set_title('<30d rate by prior inpatient visits')
d.groupby('age_mid').y30.mean().plot.bar(ax=ax[1], color='#3b6fb6'); ax[1].set_title('<30d rate by age midpoint')
d.groupby('discharge_disposition_id').y30.agg(['mean', 'size']).query('size>=500')['mean'].sort_values().plot.barh(ax=ax[2], color='#3b6fb6'); ax[2].set_title('<30d rate by discharge disposition')
for a in ax: a.set_ylabel('')
plt.tight_layout(); plt.show()
print('Base rate <30d: %.3f   any readmission: %.3f' % (d.y30.mean(), d.yany.mean()))

## 4. Patient-level split and models

In [ ]:
def split():
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED).split(X, groups=groups))
    assert set(groups.iloc[tr]).isdisjoint(groups.iloc[te])
    return tr, te

pre_lr = ColumnTransformer([('n', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
                            ('c', OneHotEncoder(handle_unknown='ignore', min_frequency=50), cat)])
pre_gb = ColumnTransformer([('n', 'passthrough', num),
                            ('c', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False), cat)])
models = {
    'Baseline (prior)': DummyClassifier(strategy='prior'),
    'Logistic regression': make_pipeline(pre_lr, LogisticRegression(C=0.05, max_iter=2000)),
    'Gradient boosting': make_pipeline(pre_gb, HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, random_state=SEED)),
}
tr, te = split()
def run(ycol):
    y = d[ycol]; out = {}; rows = []
    for n, m in models.items():
        m.fit(X.iloc[tr], y.iloc[tr]); p = m.predict_proba(X.iloc[te])[:, 1]; out[n] = p
        rows.append((n, roc_auc_score(y.iloc[te], p), average_precision_score(y.iloc[te], p), brier_score_loss(y.iloc[te], p)))
    return out, pd.DataFrame(rows, columns=['model', 'ROC-AUC', 'PR-AUC', 'Brier']).set_index('model').round(4)
P30, res30 = run('y30')
print('Readmitted <30 days. Test: %d encounters, %d patients, base rate %.3f' % (len(te), groups.iloc[te].nunique(), d.y30.iloc[te].mean()))
res30

### Secondary target: readmitted at all (`<30` or `>30`)

In [ ]:
Pany, resany = run('yany')
print('Any readmission. Test base rate %.3f' % d.yany.iloc[te].mean()); resany

## 5. Stability: grouped 5-fold CV (ROC-AUC, <30d)

In [ ]:
for n in ['Logistic regression', 'Gradient boosting']:
    s = cross_val_score(models[n], X, d.y30, groups=groups, cv=GroupKFold(5), scoring='roc_auc', n_jobs=-1)
    print(f'{n:22s} {s.mean():.3f} +/- {s.std():.3f}')

## 6. Curves, calibration and lift (<30d)

In [ ]:
yt = d.y30.iloc[te]; best = P30['Gradient boosting']
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for n, p in P30.items():
    f, t, _ = roc_curve(yt, p); ax[0].plot(f, t, label=f'{n} ({roc_auc_score(yt, p):.3f})')
    pr, rc, _ = precision_recall_curve(yt, p); ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].axhline(yt.mean(), color='k', ls=':'); ax[1].set(title='Precision-Recall', xlabel='Recall', ylabel='Precision'); ax[1].legend()
q = pd.qcut(best, 10, duplicates='drop'); cal = pd.DataFrame({'p': best, 'y': yt.values}).groupby(q).mean()
ax[2].plot(cal.p, cal.y, 'o-'); ax[2].plot([0, .3], [0, .3], 'k:'); ax[2].set(title='Calibration (deciles)', xlabel='Predicted', ylabel='Observed')
plt.tight_layout(); plt.show()

r = pd.DataFrame({'p': best, 'y': yt.values}).sort_values('p', ascending=False)
for frac in [.05, .10, .20]:
    top = r.head(int(len(r) * frac))
    print(f'Top {frac:.0%} risk: readmit rate {top.y.mean():.3f} (lift {top.y.mean() / yt.mean():.2f}x, captures {top.y.sum() / yt.sum():.0%} of <30d readmissions)')

## 7. What drives the predictions? (permutation importance, held-out ROC-AUC)

In [ ]:
m = models['Gradient boosting']
pi = permutation_importance(m, X.iloc[te], d.y30.iloc[te], scoring='roc_auc', n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(12)
imp[::-1].plot.barh(color='#3b6fb6', figsize=(7, 4.5), title='Drop in test ROC-AUC when shuffled'); plt.tight_layout(); plt.show()

## 8. Conclusions
*(Figures from a run with `SEED=0`; re-running the notebook should reproduce them.)*

**Short answer: weakly. There is real signal, but not enough for reliable individual-level prediction.**

| Target | Baseline ROC-AUC | Logistic reg. | Gradient boosting | Test base rate |
|---|---|---|---|---|
| Readmitted <30 days | 0.50 | 0.653 | 0.663 | 11.1% |
| Readmitted at all | 0.50 | 0.675 | 0.692 | 46.7% |

- Grouped 5-fold CV for <30d gives ROC-AUC 0.675 +/- 0.004 (gradient boosting) and 0.663 +/- 0.003 (logistic), consistent with the hold-out score.
- PR-AUC for <30d is 0.21 against a 0.11 base rate: about 2x better than chance, but precision is low.
- Used as a triage list, the top 10% highest-risk encounters have a 25.8% 30-day readmission rate (2.3x lift) and capture 23% of all such readmissions. That is useful for prioritising follow-up, not for confident per-patient calls.
- The main drivers are prior utilisation (`n_prior_visits`, `number_inpatient`), then discharge disposition and age. Everything else adds little, so the available columns carry limited information about readmission.

**Caveats**
- Splitting by patient matters: random row splits would leak repeat patients and give optimistic scores.
- Excluding death/hospice discharges is a modelling choice; keeping them would slightly lower apparent performance.
- Repeat encounters of the same patient are still in the training data, so the model partly learns patient history.
- Possible improvements: tuning, richer diagnosis-code features, and clinical data not in this file (labs, discharge notes, social factors). No threshold has been chosen; that needs the relative cost of missed readmissions and false alarms.
